# Final Challenge - Data Handling Workshop

## 1. Final Challenge

In this challenge you will work with a **real-world retail dataset** and apply the main
ideas of the workshop from start to finish:

**understand the data -> identify quality problems -> clean -> prepare -> analyze ->
visualize -> write a final insight**

Work through the sections in order. Each section briefly explains the task, followed by a
code cell for your work. Add more cells whenever you need them.

## 2. About the Dataset

The dataset is based on the **Online Retail** dataset. It contains transactions from a
UK-based online retail business that sells giftware; prices are in pounds sterling (GBP).
**One row is one product line on an invoice** - an invoice can have several rows.

This workshop version **may contain intentionally introduced data-quality issues**.
Finding and handling them is part of the challenge.

| Column | Meaning |
|---|---|
| `InvoiceNo` | Invoice number. Invoice numbers starting with `C` are cancellations. |
| `StockCode` | Product code |
| `Description` | Product description |
| `Quantity` | Quantity purchased (negative on cancellations) |
| `InvoiceDate` | Transaction date and time |
| `UnitPrice` | Price per unit (GBP) |
| `CustomerID` | Customer identifier |
| `Country` | Customer / transaction country |
| `TransactionType` | Purchase or cancellation |

## 3. Open the Notebook and Upload the Dataset

Both files are in the **Google Drive folder** provided with the challenge.

| Step | What to do |
|---|---|
| 1 | Open the Google Drive folder provided with the challenge. |
| 2 | Open **`Final_Challenge_Student.ipynb`** in Google Colab. |
| 3 | Save your own copy: **File -> Save a copy in Drive**. Work in your copy. |
| 4 | Download **`online_retail_challenge.csv`** from the same Google Drive folder. |
| 5 | In Colab, open the **Files** panel on the left (folder icon). |
| 6 | Click **Upload** and upload `online_retail_challenge.csv`. |
| 7 | Run the notebook from the beginning, starting with the loading cell below. |

> Files uploaded to the Colab runtime are **temporary**, so the CSV may need to be uploaded
> again if the runtime restarts.

In [ ]:
import pandas as pd

df = pd.read_csv("online_retail_challenge.csv")
df.head()

## 4. Understand the Data

Inspect the dataset before changing anything. Use appropriate tools such as `head()`,
`shape`, `columns`, `info()` and `describe()`.

Then write a few short notes: how large is the dataset, what does each column contain, and
which data types look correct or unexpected?

In [ ]:
import pandas as pd

df = pd.read_csv("online_retail_challenge.csv", dtype=str, keep_default_na=False)
print("Shape (rows, columns):", df.shape)
print("Columns:", df.columns.tolist())
df.info()
df.head()

**My notes:**
- The dataset has 2,081 rows and 9 columns. One row is one product line on an invoice.
- Every column is read as text (`dtype=str`) at first, so numbers, dates and text can all be checked.
- Unexpected types: `Quantity` and `UnitPrice` are not numeric in every row (they contain units like "pcs", currency signs `£`, and comma decimals `0,83`). `InvoiceDate` has three different formats.
- Expected types: `Quantity` and `UnitPrice` should be numbers, and `InvoiceDate` should be a date-time.

## 5. Identify Data Quality Problems

Investigate the dataset for the issues covered in the workshop, where relevant:

- missing values
- duplicates
- incorrect data types
- inconsistent values
- invalid or suspicious values

Document what you find **before** you fix anything. Remember that an unusual value is not
automatically an error - check it against the dataset context.

In [ ]:
# Quality checks (on the raw data, before any change)
import re
q_raw = df["Quantity"].str.replace(" pcs", "", regex=False)
p_raw = df["UnitPrice"].str.replace("£", "", regex=False).str.replace(",", ".", regex=False)
q = pd.to_numeric(q_raw, errors="coerce")
p = pd.to_numeric(p_raw, errors="coerce")

print("Missing (empty text):")
print((df.apply(lambda s: s.str.strip() == "")).sum())
print("\nDuplicated rows:", df.duplicated().sum())
print("\nTransactionType values:", df["TransactionType"].str.strip().value_counts().to_dict())
print("\nNon-numeric Quantity:", df.loc[q.isna(), "Quantity"].unique().tolist())
print("Non-numeric UnitPrice:", df.loc[p.isna(), "UnitPrice"].unique().tolist())
print("\nNegative UnitPrice rows:", (p < 0).sum(), "| zero Quantity rows:", (q == 0).sum())
c_invoice = df["InvoiceNo"].str.startswith("C")
print("\nCancellation invoice (C) vs label:")
print(pd.crosstab(df["TransactionType"].str.strip(), c_invoice))
print("\nCountry values:", sorted(df["Country"].str.strip().unique().tolist()))

**Problems I found:**
1. **Duplicate rows:** 10 rows are exact copies of another row (same invoice, product, quantity, price).
2. **Inconsistent `TransactionType`:** "purchase", "PURCHASE", "Cancelled", "cancellation" and values with trailing spaces, next to "Purchase" and "Cancellation". I standardized them to two labels.
3. **Wrong text in numeric columns:** `Quantity` has "pcs" (e.g., "12 pcs"), and `UnitPrice` has "£" signs and comma decimals (e.g., "0,83").
4. **Mixed date formats:** `InvoiceDate` has ISO (`2010-12-01 12:49`), day/month/year (`23/12/2010 14:06`), and month-name (`Jan 09 2011 12:46`) formats.
5. **Inconsistent country names:** "EIRE" and "Eire" (Ireland), "UK" and "UNITED KINGDOM", "GERMANY" and "germany", "france", and blank or "Unspecified" values.
6. **Invalid price:** 2 rows have a negative `UnitPrice` (-1.65 and -0.65). A product cannot have a negative price.
7. **Missing values:** `CustomerID` is blank in 146 rows, `Description` in 30, and `Country` in 17.
8. **Suspicious (kept, not errors):** 3 rows have quantity 0 (they add £0 to sales). Some quantities (e.g., 1488) and prices (e.g., 495 in an "Unspecified" country order) are large. They may be real bulk orders, so I kept them and flagged them for review.

## 6. Clean the Data

Clean the problems you discovered. Work on a copy of the data so the original stays
available.

Add **short comments** in your code to explain important decisions, for example why you
removed, corrected or kept something.

In [ ]:
# Cleaning: work on a copy, fix one problem at a time
import numpy as np
clean = df.copy()

# 1) Strip leading/trailing spaces in every text column
for col in clean.columns:
    clean[col] = clean[col].str.strip()

# 2) TransactionType: one label per meaning ("cancelled"/"cancellation" -> Cancellation)
type_map = {"purchase": "Purchase", "cancellation": "Cancellation", "cancelled": "Cancellation"}
clean["TransactionType"] = clean["TransactionType"].str.lower().map(type_map)

# 3) Numeric columns: remove units and currency, convert comma decimals
clean["Quantity"] = pd.to_numeric(clean["Quantity"].str.replace(" pcs", "", regex=False), errors="coerce")
clean["UnitPrice"] = pd.to_numeric(
    clean["UnitPrice"].str.replace("£", "", regex=False).str.replace(",", ".", regex=False), errors="coerce"
)

# 4) InvoiceDate: parse each format explicitly.
#    (dayfirst=True on mixed formats would flip ISO dates, e.g. 2010-12-01 -> 2010-01-12)
def parse_date(x):
    for fmt in ("%Y-%m-%d %H:%M", "%d/%m/%Y %H:%M", "%b %d %Y %H:%M"):
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            pass
    return pd.NaT
clean["InvoiceDate"] = clean["InvoiceDate"].map(parse_date)

# 5) Country: one spelling per country; blank / "Unspecified" -> Unknown
country_map = {
    "eire": "Ireland", "ireland": "Ireland", "uk": "United Kingdom", "united kingdom": "United Kingdom",
    "germany": "Germany", "france": "France", "usa": "USA", "channel islands": "Channel Islands",
    "unspecified": "Unknown", "": "Unknown",
}
clean["Country"] = clean["Country"].map(lambda x: country_map.get(x.lower(), x))

# 6) Exact duplicate rows: keep one copy
n_before = len(clean)
clean = clean.drop_duplicates()
print("Duplicates removed:", n_before - len(clean))

# 7) Negative prices are invalid: remove those rows
bad_price = clean["UnitPrice"] < 0
print("Negative-price rows removed:", bad_price.sum())
clean = clean[~bad_price].reset_index(drop=True)

# 8) Check: cancellation label matches invoice prefix "C" and quantity sign
c_inv = clean["InvoiceNo"].str.startswith("C")
assert ((clean["TransactionType"] == "Cancellation") == c_inv).all()
assert (clean.loc[clean["TransactionType"] == "Cancellation", "Quantity"] < 0).all()

print("Rows before:", len(df), "| after:", len(clean))
print("Missing values after cleaning:")
print(clean.isna().sum())
clean.dtypes

## 7. Create TotalAmount

Create a useful derived column called **`TotalAmount`** that holds the total value of each
transaction line. Create it only after the columns it depends on are clean, and check that
the result looks sensible.

In [ ]:
# TotalAmount = Quantity x UnitPrice (after Quantity and UnitPrice are numeric)
clean["TotalAmount"] = (clean["Quantity"] * clean["UnitPrice"]).round(2)

# Sanity check: cancellations should be negative, purchases positive
print(clean.groupby("TransactionType")["TotalAmount"].agg(["count", "sum", "min", "max"]).round(2))
clean[["InvoiceNo", "Quantity", "UnitPrice", "TotalAmount", "TransactionType"]].head()

## 8. Analysis

Calculate the **Top 5 countries by total sales**.

In [ ]:
# Top 5 countries by total sales (net of cancellations)
# Rows with Country = "Unknown" are excluded because we cannot assign them to a country.
known = clean[clean["Country"] != "Unknown"]
top5 = (
    known.groupby("Country")["TotalAmount"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .round(2)
)
share = (top5 / known["TotalAmount"].sum() * 100).round(1)
result = pd.DataFrame({"TotalSales (£)": top5, "Share of net sales (%)": share})
result

## 9. Visualization

Choose and create an appropriate visualization for the Top 5 countries result.
The chart should have a **clear title**, **readable labels** and an **appropriate chart
type**.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(top5.index, top5.values, color="#2E75B6")
for bar, value in zip(bars, top5.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f"£{value:,.0f}", ha="center", va="bottom", fontsize=9)

ax.set_title("Top 5 Countries by Total Sales (net of cancellations)", fontsize=13, pad=12)
ax.set_xlabel("Country")
ax.set_ylabel("Total sales (GBP £)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("£{x:,.0f}"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 10. Final Insight

Write **1-3 concise observations** based on your result.

An observation must be supported by the data. Do not claim a cause (*"X happened because
of Y"*) unless your data actually shows it.

**My observations:**
1. **The United Kingdom dominates:** UK sales are about £28,034, which is 47.4% of net sales. The top 5 countries together make up 78.0% of net sales (£46,177 of £59,168).
2. **Cancellations are material:** cancellations total about -£7,550, which is 11.3% of gross purchases (about £66,718). This is a large share, and it reduces the net sales used for the ranking.
3. **Japan's rank depends on one order:** Japan is #2 with £5,671, but one order (invoice 537899, quantity 1,488 at £2.55, about £3,794) is 66.9% of Japan's total. Without that order Japan would be about £1,877 (18 rows in total). The result depends on that single large order, so it should be checked before drawing conclusions about Japan.

## 11. Optional Bonus

These tasks are **optional**. Do them only after finishing sections 1-10.

- **A.** Answer one additional analysis question of your choice.
- **B.** Create one additional visualization.
- **C.** Investigate one unusual value: is it a data mistake or real business data?
  Compare it with related rows before you decide.
- **D.** Export your cleaned dataset as `retail_prepared.csv` (without the index column).

In [ ]:
# Optional bonus D: export the cleaned dataset (without the index column)
clean.to_csv("retail_prepared.csv", index=False)
print("Saved retail_prepared.csv with", len(clean), "rows")